### 1- Load the dataset

In [ ]:
from pathlib import Path
import json

import pandas as pd


DATA_DIR = Path("../data/raw")

NODES_PATH = DATA_DIR / "isebel-mecklenburg-nodes.csv"
EDGES_PATH = DATA_DIR / "isebel-mecklenburg-edges.csv"

nodes = pd.read_csv(NODES_PATH)
edges = pd.read_csv(EDGES_PATH)

print("Nodes shape:", nodes.shape)
print("Edges shape:", edges.shape)

print("\nNode columns:")
print(nodes.columns.tolist())

print("\nEdge columns:")
print(edges.columns.tolist())

display(nodes.head())
display(edges.head())

Nodes shape: (20005, 3)
Edges shape: (40604, 4)

Node columns:
['id', 'label', 'properties']

Edge columns:
['src-id', 'dst-id', 'label', 'properties']


,id,label,properties
0,12495,date,"{""datetime"":""1938-10-24"",""__unique"":[""datetime..."
1,10923,date,"{""datetime"":""1925-09-06"",""__unique"":[""datetime..."
2,14299,date,"{""datetime"":""1927-07-12"",""__unique"":[""datetime..."
3,4957,date,"{""datetime"":""1923-09-03"",""__unique"":[""datetime..."
4,2835,date,"{""datetime"":""1935-08-20"",""__unique"":[""datetime..."


,src-id,dst-id,label,properties
0,16006,284,date-of-narration,"{""__label"":""date-of-narration""}"
1,18144,628,date-of-narration,"{""__label"":""date-of-narration""}"
2,3514,1719,date-of-narration,"{""__label"":""date-of-narration""}"
3,3801,3536,date-of-narration,"{""__label"":""date-of-narration""}"
4,49,56,date-of-narration,"{""__label"":""date-of-narration""}"


In [ ]:
print("Unique node labels:")
print(sorted(nodes["label"].dropna().unique()))

print("\nNumber of unique node labels:")
print(nodes["label"].nunique())

print("\nUnique edge labels:")
print(sorted(edges["label"].dropna().unique()))

print("\nNumber of unique edge labels:")
print(edges["label"].nunique())

Unique node labels:
['date', 'keyword', 'person', 'place', 'story']

Number of unique node labels:
5

Unique edge labels:
['content', 'date-of-narration', 'narrator', 'place-of-narration']

Number of unique edge labels:
4


In [ ]:
node_type_counts = (
    nodes["label"]
    .value_counts(dropna=False)
    .rename_axis("node_type")
    .reset_index(name="count")
)

node_type_counts["percentage"] = (
    node_type_counts["count"] / len(nodes) * 100
).round(2)


edge_type_counts = (
    edges["label"]
    .value_counts(dropna=False)
    .rename_axis("edge_type")
    .reset_index(name="count")
)

edge_type_counts["percentage"] = (
    edge_type_counts["count"] / len(edges) * 100
).round(2)

print("Node types:")
display(node_type_counts)

print("Relationship types:")
display(edge_type_counts)

Node types:


,node_type,count,percentage
0,story,14379,71.88
1,keyword,2439,12.19
2,place,1393,6.96
3,person,1088,5.44
4,date,706,3.53


Relationship types:


,edge_type,count,percentage
0,content,30410,74.89
1,place-of-narration,5550,13.67
2,narrator,2365,5.82
3,date-of-narration,2279,5.61


In [ ]:
edge_label_counts = (
    edges["label"]
    .value_counts(dropna=False)
    .rename_axis("edge_type")
    .reset_index(name="count")
)

edge_label_counts["percentage"] = (
    edge_label_counts["count"] / len(edges) * 100
).round(2)

display(edge_label_counts)

,edge_type,count,percentage
0,content,30410,74.89
1,place-of-narration,5550,13.67
2,narrator,2365,5.82
3,date-of-narration,2279,5.61


In [ ]:
node_samples = (
    nodes
    .groupby("label", group_keys=False)
    .head(3)
    .sort_values("label")
)

display(node_samples)

,id,label,properties
0,12495,date,"{""datetime"":""1938-10-24"",""__unique"":[""datetime..."
1,10923,date,"{""datetime"":""1925-09-06"",""__unique"":[""datetime..."
2,14299,date,"{""datetime"":""1927-07-12"",""__unique"":[""datetime..."
3187,4867,keyword,"{""name"":""Tagelöhner"",""__unique"":[""name""],""__la..."
3188,928,keyword,"{""name"":""buttern"",""__unique"":[""name""],""__label..."
3189,3914,keyword,"{""name"":""Fear"",""__unique"":[""name""],""__label"":""..."
706,1589,person,"{""name"":""Schumacher"",""gender"":""male"",""__unique..."
707,6581,person,"{""name"":""Doss"",""gender"":""male"",""__unique"":[""ge..."
708,16737,person,"{""name"":""Jörn"",""gender"":""female"",""__unique"":[""..."
1794,1167,place,"{""name"":""England"",""long"":"""",""lat"":"""",""__unique..."


In [ ]:
edges_samples = (
    edges
    .groupby("label", group_keys=False)
    .head(3)
    .sort_values("label")
)

display(edges_samples)

,src-id,dst-id,label,properties
10194,1319,931,content,"{""__label"":""content""}"
10195,1453,30,content,"{""__label"":""content""}"
10196,15342,206,content,"{""__label"":""content""}"
0,16006,284,date-of-narration,"{""__label"":""date-of-narration""}"
1,18144,628,date-of-narration,"{""__label"":""date-of-narration""}"
2,3514,1719,date-of-narration,"{""__label"":""date-of-narration""}"
2279,5975,5976,narrator,"{""__label"":""narrator""}"
2280,14492,1331,narrator,"{""__label"":""narrator""}"
2281,14497,14498,narrator,"{""__label"":""narrator""}"
4644,13978,2275,place-of-narration,"{""__label"":""place-of-narration""}"


In [ ]:
def parse_properties(value):
    """Convert a JSON property string into a Python dictionary."""
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:
        return json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return {"_unparsed_value": value}


nodes = nodes.copy()
edges = edges.copy()

nodes["properties_dict"] = nodes["properties"].apply(parse_properties)
edges["properties_dict"] = edges["properties"].apply(parse_properties)

In [ ]:
failed_node_parsing = nodes["properties_dict"].apply(
    lambda value: "_unparsed_value" in value
).sum()

failed_edge_parsing = edges["properties_dict"].apply(
    lambda value: "_unparsed_value" in value
).sum()

print("Unparsed node-property rows:", failed_node_parsing)
print("Unparsed edge-property rows:", failed_edge_parsing)

Unparsed node-property rows: 0
Unparsed edge-property rows: 0


In [ ]:
def collect_property_keys(series):
    """Return all property names used in a series of dictionaries."""
    property_keys = set()

    for properties in series:
        if isinstance(properties, dict):
            property_keys.update(properties.keys())

    return sorted(property_keys)


node_property_keys = (
    nodes
    .groupby("label")["properties_dict"]
    .apply(collect_property_keys)
    .reset_index(name="property_keys")
)

display(node_property_keys)

,label,property_keys
0,date,"[__label, __unique, datetime]"
1,keyword,"[__label, __unique, name]"
2,person,"[__label, __unique, gender, name]"
3,place,"[__label, __unique, lat, long, name]"
4,story,"[__label, __unique, path, title]"


In [ ]:
edge_property_keys = (
    edges
    .groupby("label")["properties_dict"]
    .apply(collect_property_keys)
    .reset_index(name="property_keys")
)

display(edge_property_keys)

,label,property_keys
0,content,[__label]
1,date-of-narration,[__label]
2,narrator,[__label]
3,place-of-narration,[__label]


In [ ]:
# Use strings to avoid matching problems caused by mixed integer/string IDs.
nodes["id_key"] = nodes["id"].astype(str)
edges["src_key"] = edges["src-id"].astype(str)
edges["dst_key"] = edges["dst-id"].astype(str)

id_to_node_type = nodes.set_index("id_key")["label"]

edges["src_type"] = edges["src_key"].map(id_to_node_type)
edges["dst_type"] = edges["dst_key"].map(id_to_node_type)

display(
    edges[
        ["src-id", "src_type", "label", "dst-id", "dst_type"]
    ].head(10)
)

,src-id,src_type,label,dst-id,dst_type
0,16006,story,date-of-narration,284,date
1,18144,story,date-of-narration,628,date
2,3514,story,date-of-narration,1719,date
3,3801,story,date-of-narration,3536,date
4,49,story,date-of-narration,56,date
5,11294,story,date-of-narration,1377,date
6,11339,story,date-of-narration,8992,date
7,14072,story,date-of-narration,3665,date
8,2545,story,date-of-narration,2550,date
9,17583,story,date-of-narration,1606,date


In [ ]:
relationship_schema = (
    edges
    .groupby(
        ["src_type", "label", "dst_type"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(relationship_schema)

,src_type,label,dst_type,count
0,story,content,keyword,30410
3,story,place-of-narration,place,5550
2,story,narrator,person,2365
1,story,date-of-narration,date,2279


In [ ]:
missing_src = edges["src_type"].isna().sum()
missing_dst = edges["dst_type"].isna().sum()

print("Edges with unknown source node:", missing_src)
print("Edges with unknown destination node:", missing_dst)

print(
    "Edges with at least one unknown endpoint:",
    (edges["src_type"].isna() | edges["dst_type"].isna()).sum()
)

Edges with unknown source node: 0
Edges with unknown destination node: 0
Edges with at least one unknown endpoint: 0


In [ ]:
for node_type in sorted(nodes["label"].dropna().unique()):
    print(f"\n=== {node_type} ===")

    examples = (
        nodes.loc[
            nodes["label"] == node_type,
            ["id", "properties_dict"]
        ]
        .head(5)
    )

    display(examples)


=== date ===


,id,properties_dict
0,12495,"{'datetime': '1938-10-24', '__unique': ['datet..."
1,10923,"{'datetime': '1925-09-06', '__unique': ['datet..."
2,14299,"{'datetime': '1927-07-12', '__unique': ['datet..."
3,4957,"{'datetime': '1923-09-03', '__unique': ['datet..."
4,2835,"{'datetime': '1935-08-20', '__unique': ['datet..."



=== keyword ===


,id,properties_dict
3187,4867,"{'name': 'Tagelöhner', '__unique': ['name'], '..."
3188,928,"{'name': 'buttern', '__unique': ['name'], '__l..."
3189,3914,"{'name': 'Fear', '__unique': ['name'], '__labe..."
3190,7690,"{'name': 'verletzen', '__unique': ['name'], '_..."
3191,5873,"{'name': 'Pfosten', '__unique': ['name'], '__l..."



=== person ===


,id,properties_dict
706,1589,"{'name': 'Schumacher', 'gender': 'male', '__un..."
707,6581,"{'name': 'Doss', 'gender': 'male', '__unique':..."
708,16737,"{'name': 'Jörn', 'gender': 'female', '__unique..."
709,11902,"{'name': 'Hinrichs', 'gender': 'male', '__uniq..."
710,14432,"{'name': 'Behnick', 'gender': 'male', '__uniqu..."



=== place ===


,id,properties_dict
1794,1167,"{'name': 'England', 'long': '', 'lat': '', '__..."
1795,2741,"{'name': 'Pritzier', 'long': 'OOPS', 'lat': 'O..."
1796,19779,"{'name': 'Hohen Lukow', 'long': '11.96195', 'l..."
1797,2324,"{'name': 'Börgerende', 'long': 'OOPS', 'lat': ..."
1798,4150,"{'name': 'Lüningsdorf', 'long': 'OOPS', 'lat':..."



=== story ===


,id,properties_dict
5626,16368,"{'path': 'de.wossidia.xmd-s001-000-007-966', '..."
5627,14282,"{'path': 'de.wossidia.xmd-s001-000-005-437', '..."
5628,8041,"{'path': 'de.wossidia.xmd-s001-000-006-499', '..."
5629,11481,"{'path': 'de.wossidia.xmd-s001-000-014-227', '..."
5630,12699,"{'path': 'de.wossidia.xmd-s001-000-001-163', '..."


In [ ]:
date_nodes = nodes[nodes["label"] == "date"].copy()

date_nodes["datetime"] = date_nodes["properties_dict"].apply(
    lambda properties: properties.get("datetime")
)

display(date_nodes[["id", "datetime"]].head(10))

print("Earliest date:", date_nodes["datetime"].min())
print("Latest date:", date_nodes["datetime"].max())
print("Unique dates:", date_nodes["datetime"].nunique())

,id,datetime
0,12495,1938-10-24
1,10923,1925-09-06
2,14299,1927-07-12
3,4957,1923-09-03
4,2835,1935-08-20
5,7123,1914-04-07
6,8166,1935-09-16
7,6714,1910-12-30
8,17796,1916-08-05
9,17260,1923-03-31


Earliest date: 1016-08-12
Latest date: 2014-04-08
Unique dates: 706


In [ ]:
date_nodes = nodes[nodes["label"] == "date"].copy()

date_nodes["datetime_raw"] = date_nodes["properties_dict"].apply(
    lambda p: p.get("datetime")
)

date_nodes["datetime"] = pd.to_datetime(
    date_nodes["datetime_raw"],
    errors="coerce"
)

date_nodes["year"] = date_nodes["datetime"].dt.year

print("Missing or invalid dates:", date_nodes["datetime"].isna().sum())
print("Earliest parsed date:", date_nodes["datetime"].min())
print("Latest parsed date:", date_nodes["datetime"].max())

display(
    date_nodes.sort_values("datetime")[
        ["id", "datetime_raw", "datetime"]
    ].head(20)
)

Missing or invalid dates: 0
Earliest parsed date: 1016-08-12 00:00:00
Latest parsed date: 2014-04-08 00:00:00


,id,datetime_raw,datetime
420,12427,1016-08-12,1016-08-12
500,7809,1643-01-01,1643-01-01
160,15209,1683-01-01,1683-01-01
41,7505,1825-01-01,1825-01-01
186,594,1848-01-01,1848-01-01
680,10085,1860-11-20,1860-11-20
312,7674,1870-01-01,1870-01-01
159,3565,1884-01-01,1884-01-01
397,2272,1885-01-01,1885-01-01
104,3856,1887-01-01,1887-01-01


In [ ]:
suspicious_dates = date_nodes[
    (date_nodes["year"] < 1800) |
    (date_nodes["year"] > pd.Timestamp.today().year)
]

display(
    suspicious_dates[
        ["id", "datetime_raw", "datetime"]
    ].sort_values("datetime")
)

,id,datetime_raw,datetime
420,12427,1016-08-12,1016-08-12
500,7809,1643-01-01,1643-01-01
160,15209,1683-01-01,1683-01-01


In [ ]:
place_nodes = nodes[nodes["label"] == "place"].copy()

place_nodes["name"] = place_nodes["properties_dict"].apply(
    lambda p: p.get("name")
)

place_nodes["longitude_raw"] = place_nodes["properties_dict"].apply(
    lambda p: p.get("long")
)

place_nodes["latitude_raw"] = place_nodes["properties_dict"].apply(
    lambda p: p.get("lat")
)

place_nodes["longitude"] = pd.to_numeric(
    place_nodes["longitude_raw"],
    errors="coerce"
)

place_nodes["latitude"] = pd.to_numeric(
    place_nodes["latitude_raw"],
    errors="coerce"
)

print("Total places:", len(place_nodes))
print("Places with valid coordinates:", (
    place_nodes["longitude"].notna() &
    place_nodes["latitude"].notna()
).sum())

print("Places missing usable coordinates:", (
    place_nodes["longitude"].isna() |
    place_nodes["latitude"].isna()
).sum())

Total places: 1393
Places with valid coordinates: 636
Places missing usable coordinates: 757


In [ ]:
print("Node IDs unique:", nodes["id"].is_unique)
print("Duplicated node IDs:", nodes["id"].duplicated().sum())

print(
    "Exact duplicated node rows:",
    nodes.duplicated(
        subset=["id", "label", "properties"]
    ).sum()
)

print(
    "Duplicated relationships:",
    edges.duplicated(
        subset=["src-id", "dst-id", "label"]
    ).sum()
)

print(
    "Exact duplicated edge rows:",
    edges.duplicated(
        subset=["src-id", "dst-id", "label", "properties"]
    ).sum()
)

Node IDs unique: True
Duplicated node IDs: 0
Exact duplicated node rows: 0
Duplicated relationships: 0
Exact duplicated edge rows: 0


In [ ]:
print("Missing node IDs:", nodes["id"].isna().sum())
print("Missing node labels:", nodes["label"].isna().sum())
print("Missing node properties:", nodes["properties"].isna().sum())

print("Missing source IDs:", edges["src-id"].isna().sum())
print("Missing destination IDs:", edges["dst-id"].isna().sum())
print("Missing edge labels:", edges["label"].isna().sum())

Missing node IDs: 0
Missing node labels: 0
Missing node properties: 0
Missing source IDs: 0
Missing destination IDs: 0
Missing edge labels: 0


In [ ]:
failed_node_parsing = nodes["properties_dict"].apply(
    lambda p: "_unparsed_value" in p
).sum()

failed_edge_parsing = edges["properties_dict"].apply(
    lambda p: "_unparsed_value" in p
).sum()

print("Unparsed node-property rows:", failed_node_parsing)
print("Unparsed edge-property rows:", failed_edge_parsing)

Unparsed node-property rows: 0
Unparsed edge-property rows: 0


In [ ]:
story_ids = nodes.loc[
    nodes["label"] == "story",
    "id"
]

story_coverage = (
    edges
    .pivot_table(
        index="src-id",
        columns="label",
        values="dst-id",
        aggfunc="nunique",
        fill_value=0
    )
    .reindex(story_ids, fill_value=0)
)

story_coverage.index.name = "story_id"

display(story_coverage.head())

label,content,date-of-narration,narrator,place-of-narration
story_id,,,,
16368,0,0,0,0
14282,0,0,0,0
8041,5,1,0,0
11481,0,0,0,0
12699,7,1,0,5


In [ ]:
coverage_summary = pd.DataFrame({
    "stories_with_relation": (story_coverage > 0).sum(),
    "percentage_of_stories": (
        (story_coverage > 0).mean() * 100
    ).round(2),
    "average_per_story": story_coverage.mean().round(2),
    "maximum_per_story": story_coverage.max()
})

display(coverage_summary)

,stories_with_relation,percentage_of_stories,average_per_story,maximum_per_story
label,,,,
content,4577,31.83,2.11,40
date-of-narration,2270,15.79,0.16,2
narrator,2328,16.19,0.16,3
place-of-narration,2652,18.44,0.39,8


In [ ]:
stories_without_keywords = story_coverage[
    story_coverage.get("content", 0) == 0
]

print("Stories without keywords:", len(stories_without_keywords))

Stories without keywords: 9802


In [ ]:
content_edges = edges[edges["label"] == "content"]

keyword_story_counts = (
    content_edges
    .groupby("dst-id")["src-id"]
    .nunique()
    .sort_values(ascending=False)
)

display(keyword_story_counts.describe())

print(
    "Keywords appearing in one story:",
    (keyword_story_counts == 1).sum()
)

print(
    "Keywords appearing in at least 100 stories:",
    (keyword_story_counts >= 100).sum()
)

count    2439.000000
mean       12.468225
std        48.884527
min         1.000000
25%         1.000000
50%         2.000000
75%         7.000000
max       714.000000
Name: src-id, dtype: float64

Keywords appearing in one story: 998
Keywords appearing in at least 100 stories: 41


In [ ]:
keyword_names = (
    nodes[nodes["label"] == "keyword"]
    .set_index("id")["properties_dict"]
    .apply(lambda p: p.get("name"))
)

keyword_frequency_table = (
    keyword_story_counts
    .rename("story_count")
    .to_frame()
    .join(keyword_names.rename("keyword"))
    .reset_index(names="keyword_id")
)

display(
    keyword_frequency_table[
        ["keyword_id", "keyword", "story_count"]
    ].head(30)
)

,keyword_id,keyword,story_count
0,16,Frevelsage,714
1,18,Frevel,714
2,17,sacrilege,714
3,32,Hexe,628
4,35,witches,626
5,29,Hexen,626
6,92,Tote,588
7,91,dead,588
8,93,Toter,557
9,6,historical legends,516


In [ ]:
content_edges = edges[edges["label"] == "content"].copy()

keyword_to_stories = (
    content_edges
    .groupby("dst-id")["src-id"]
    .apply(frozenset)
)

identical_keyword_groups = {}

for keyword_id, story_set in keyword_to_stories.items():
    identical_keyword_groups.setdefault(story_set, []).append(keyword_id)

identical_keyword_groups = [
    keyword_ids
    for keyword_ids in identical_keyword_groups.values()
    if len(keyword_ids) > 1
]

rows = []

for group_number, keyword_ids in enumerate(
    identical_keyword_groups,
    start=1
):
    for keyword_id in keyword_ids:
        rows.append({
            "group": group_number,
            "keyword_id": keyword_id,
            "keyword": keyword_names.get(keyword_id),
            "story_count": len(keyword_to_stories[keyword_id])
        })

identical_keyword_table = pd.DataFrame(rows)

display(
    identical_keyword_table
    .sort_values(["story_count", "group"], ascending=[False, True])
    .head(100)
)

,group,keyword_id,keyword,story_count
4,3,16,Frevelsage,714
5,3,17,sacrilege,714
6,3,18,Frevel,714
7,4,29,Hexen,626
8,4,35,witches,626
...,...,...,...,...
303,120,1775,nicht grugen,33
304,120,1776,not to fear,33
35,17,140,Wann,32
36,17,146,When,32


In [ ]:
stories = nodes[nodes["label"] == "story"].copy()
stories["path"] = stories["properties_dict"].apply(
    lambda p: p.get("path")
)
stories["title"] = stories["properties_dict"].apply(
    lambda p: p.get("title")
)

keywords = nodes[nodes["label"] == "keyword"].copy()
keywords["name"] = keywords["properties_dict"].apply(
    lambda p: p.get("name")
)

people = nodes[nodes["label"] == "person"].copy()
people["name"] = people["properties_dict"].apply(
    lambda p: p.get("name")
)
people["gender"] = people["properties_dict"].apply(
    lambda p: p.get("gender")
)

In [ ]:
valid_place_mask = (
    place_nodes["longitude"].notna()
    & place_nodes["latitude"].notna()
    & place_nodes["longitude"].between(-180, 180)
    & place_nodes["latitude"].between(-90, 90)
)

valid_place_ids = set(
    place_nodes.loc[valid_place_mask, "id"]
)

place_edges = edges[
    edges["label"] == "place-of-narration"
].copy()

stories_with_any_place = set(place_edges["src-id"])

stories_with_valid_coordinates = set(
    place_edges.loc[
        place_edges["dst-id"].isin(valid_place_ids),
        "src-id"
    ]
)

total_stories = len(story_ids)

print("Stories with any place:", len(stories_with_any_place))
print(
    "Percentage with any place:",
    round(len(stories_with_any_place) / total_stories * 100, 2)
)

print(
    "Stories with at least one valid coordinate:",
    len(stories_with_valid_coordinates)
)

print(
    "Percentage with valid coordinates:",
    round(
        len(stories_with_valid_coordinates)
        / total_stories * 100,
        2
    )
)

if stories_with_any_place:
    print(
        "Valid-coordinate coverage among stories with places:",
        round(
            len(stories_with_valid_coordinates)
            / len(stories_with_any_place) * 100,
            2
        )
    )

Stories with any place: 2652
Percentage with any place: 18.44
Stories with at least one valid coordinate: 2390
Percentage with valid coordinates: 16.62
Valid-coordinate coverage among stories with places: 90.12


In [ ]:
stories_with_keywords = set(
    story_coverage.index[
        story_coverage["content"] > 0
    ]
)

geographic_analysis_stories = (
    stories_with_keywords
    & stories_with_valid_coordinates
)

print(
    "Stories usable for both community and geographic analysis:",
    len(geographic_analysis_stories)
)

print(
    "Percentage of keyword-annotated stories with coordinates:",
    round(
        len(geographic_analysis_stories)
        / len(stories_with_keywords) * 100,
        2
    )
)

Stories usable for both community and geographic analysis: 2384
Percentage of keyword-annotated stories with coordinates: 52.09


In [ ]:
for relation in [
    "date-of-narration",
    "narrator",
    "place-of-narration"
]:
    multiple = story_coverage[
        story_coverage[relation] > 1
    ]

    print(
        f"{relation}: "
        f"{len(multiple)} stories have multiple values"
    )

    display(
        multiple[[relation]]
        .sort_values(relation, ascending=False)
        .head(10)
    )

date-of-narration: 9 stories have multiple values


label,date-of-narration
story_id,
6976,2
6910,2
1022,2
575,2
19227,2
17431,2
1945,2
8130,2
15894,2


narrator: 35 stories have multiple values


label,narrator
story_id,
12564,3
13465,3
12584,2
12236,2
6224,2
5429,2
11334,2
660,2
16240,2


place-of-narration: 2070 stories have multiple values


label,place-of-narration
story_id,
7189,8
15821,7
575,7
16559,7
19988,6
16832,6
3339,6
4143,6
4509,6


In [ ]:
story_details = stories[
    ["id", "path", "title"]
].rename(columns={"id": "story_id"})

suspicious_date_details = (
    edges.loc[
        edges["label"] == "date-of-narration",
        ["src-id", "dst-id"]
    ]
    .merge(
        suspicious_dates[
            ["id", "datetime_raw"]
        ],
        left_on="dst-id",
        right_on="id",
        how="inner"
    )
    .merge(
        story_details,
        left_on="src-id",
        right_on="story_id",
        how="left"
    )
)

display(
    suspicious_date_details[
        [
            "story_id",
            "title",
            "path",
            "datetime_raw"
        ]
    ]
)

,story_id,title,path,datetime_raw
0,10189,[12520] Hexen - Blocksberg Varia,de.wossidia.xmd-s001-000-012-520,1643-01-01
1,15205,[10798] Hexen - Buhle,de.wossidia.xmd-s001-000-010-798,1683-01-01
2,12426,[10819] Hexen - Blocksberg Orte,de.wossidia.xmd-s001-000-010-819,1016-08-12
3,7807,[12518] Hexen - Blocksberg Varia,de.wossidia.xmd-s001-000-012-518,1643-01-01


In [ ]:
stories["title_clean"] = (
    stories["title"]
    .fillna("")
    .astype(str)
    .str.strip()
)

stories["path_clean"] = (
    stories["path"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print(
    "Stories without titles:",
    stories["title_clean"].eq("").sum()
)

print(
    "Stories without paths:",
    stories["path_clean"].eq("").sum()
)

print(
    "Duplicated non-empty paths:",
    stories.loc[
        stories["path_clean"].ne(""),
        "path_clean"
    ].duplicated().sum()
)

print(
    "Duplicated non-empty titles:",
    stories.loc[
        stories["title_clean"].ne(""),
        "title_clean"
    ].duplicated().sum()
)

Stories without titles: 8
Stories without paths: 0
Duplicated non-empty paths: 0
Duplicated non-empty titles: 0
